In [1]:
import pynucastro as pyna
import numpy as np
from pathlib import Path

In [2]:
rates=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_18_9_20")
rates_exp=pyna.rates.library.Library(libfile=r"Nuclear_data\decays\actual\Reaclib_default_Experimental")

sources_label_default=[]
for r in rates.get_rates():
    if r.source['Label'] not in sources_label_default:
        sources_label_default.append(r.source['Label'])
        

sources_label_experimental=[]
for r in rates_exp.get_rates():
    if r.source['Label'] not in sources_label_experimental:
        sources_label_experimental.append(r.source['Label'])
        

sources_label_theoretical=[]
for rate in rates.get_rates():
    if rate.source['Label'] not in sources_label_experimental:
        if rate.source['Label']  not in sources_label_theoretical:
            sources_label_theoretical.append(rate.source['Label'] )
            


In [ ]:
def parse_rate_line(line):
    # Extract fields by fixed positions
    nuclides = [line[5 + i*5 : 10 + i*5].strip() for i in range(6)]
    set_label = line[43:47].strip()
    rate_flag = line[47].strip()  # 'n', 'r', or 'w'
    reverse_flag = line[48].strip()  # 'v' if reverse
    q_value = float(line[52:64].strip())
    return {
        "nuclides": nuclides,
        "set_label": set_label,
        "rate_flag": rate_flag,
        "reverse_flag": reverse_flag,
        "q_value": q_value
    }

def is_alpha_decay(line):
    """Check if Reaclib header line corresponds to an alpha decay."""
    nuclides = [line[5 + i*5 : 10 + i*5].strip().lower() for i in range(6)]
    nuclides = [n for n in nuclides if n]
    # Usually products are at the end
    return len(nuclides) ==3 and nuclides[1] in ("he4", "4he", "a")


def zero_coefficients_line(line1,line2):
    coeffients1=[float(line1[i*13:+13+i*13].strip()) for i in range(4)]
    coeffients2=[float(line2[i*13:+13+i*13].strip()) for i in range(2)]
   
    new=sum(np.array(coeffients1))+sum(np.array(coeffients2))
    """Replace all coefficients (scientific notation values) with 0.00000e+00."""
    if new<0:
        return f"{new:.5e}"+" 0.00000e+00"*3 + " "*23+'\n'
    else:
        return f" {new:.5e}"+" 0.00000e+00"*3 + " "*23+'\n'
    

def process_reaclib(infile, outfile):
    """Reads a Reaclib file and zeros out coefficients for alpha decays."""
    with open(infile, "r") as fin, open(outfile, "w") as fout:
        lines = fin.readlines()
        i = 0
        chapter=False
        while i < len(lines):
            line = lines[i]
            if line[0]=='2':
                chapter=True
            elif line[0]!=' ' or line[0]!='-':
                chapter=False
            if line[0:5] == "     " and line[0:10]!='          ':
                # This is a header line
                fout.write(line)
                if is_alpha_decay(line) and chapter:
                    if i + 3 < len(lines):
                        fout.write(zero_coefficients_line(lines[i + 1],lines[i+2]))
                        fout.write(" 0.00000e+00"*3 + " "*36+'\n')
                    i += 3
                    continue
                
            
            else:
                fout.write(line)
            i += 1

In [13]:

output_file=Path('Nuclear_data/decays/reaclib_alpha_T_1GK') 
input_file=Path(r'Nuclear_data\decays\actual\Reaclib_18_9_20')
process_reaclib(input_file,output_file)